[Guide Index](README.md)

# Appendix D: How StarLayer classifies errors

When something goes wrong, StarLayer raises an error from one small family of classes. This appendix explains the family in plain terms: how to tell which error you will get, how to catch it, and what changed from the earlier release. It ends with the open questions about the scheme, for review.

The whole idea fits in three rules:

1. **A wrong argument is a `ValueError` or a `TypeError`.** You called a function with something it refuses. Change the argument and the error goes away.
2. **Anything else StarLayer raises on purpose is one of four classes, and the class says what kind of problem it is.** The text or data is malformed (`FormatError`), the shapes are wrong (`ShapeError`), the work could not be done (`ExecutionError`), or a SPARQL endpoint failed (`EndpointError`).
3. **The class names the cause, and a `kind` says which cause.** There is one `FormatError`, not one class per language, and its `kind` says whether the malformed thing was SPARQL, SRL, RDF, a term, Manchester Syntax or OWL. Where it happened (the function, the document) is the `context`, never a different class.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). Run cells from top to bottom. The setup cell defines `show()`, which runs a call that is expected to fail and prints the error's class, its `kind`, its `context` and its message.

In [ ]:
from starlayer.errors import ExecutionError, FormatError, ShapeError, StarLayerError, only_kind
from starlayer.graph import XSD, StarLayerGraph, literal, parse_term, to_python
from starlayer.shacl import meta_validate
from starlayer.sparql import sparql_to_tree, srl, tree_to_sparql
from starlayer.starontology.manchester import manchester_to_tree


def show(label, call):
    """Run a call that is expected to fail, and print what was raised."""
    try:
        call()
    except Exception as error:
        kind = getattr(error, "kind", None)
        position = f" line {error.line}" if getattr(error, "line", None) else ""
        context = getattr(error, "context", None)
        print(f"{label}")
        print(f"   {type(error).__name__}, kind={kind!r}, context={context!r}{position}")
        message = getattr(error, "why", None) or str(error).splitlines()[0]
        print(f"   {message[:100]}")
    else:
        print(label, "-> no error")


g = StarLayerGraph()
g.parse(data="@prefix ex: <http://example.org/> . ex:a ex:p ex:b .", format="turtle")

<StarLayerGraph DEFAULT (1 triples)>

## 1. The family

```
StarLayerError                  everything StarLayer raises on purpose
 |- FormatError                 text or data is malformed             kind, context, line, col
 |- ExecutionError              the work could not be done            kind, context
 |- ShapeError                  a shapes graph is wrong               context
 |   |- ShapeCompileError       the engine cannot read or compile it
 |   `- MetaShaclError          the meta check found the problem      problems, report_text
 `- EndpointError               a SPARQL endpoint failed              status (also an OSError)
```

`except StarLayerError` catches all of them. The classes are not a `ValueError`, a `SyntaxError` or a `RuntimeError`, so `except ValueError` no longer catches a malformed document or a wrong shapes graph. (`EndpointError` is also an `OSError`, because a network failure is an I/O error.)

## 2. Which error will I get?

Ask three questions, in this order. The first one that fits decides the class.

1. **Is the argument the wrong Python type?** A decimal where text was expected, a number where a graph was expected. That is a **`TypeError`**.
2. **Is it a thing with a defined format, and its content cannot be read as that format?** Text that should be SPARQL, a Turtle document, the text of a term, a tree, a rule set. The type is right, but the content does not fit. That is a **`FormatError`**.
3. **Is it a parameter or an object that the call cannot use as it stands?** A value outside the choices the call defines, options that contradict each other, an object in a state the call refuses (a closed one), the wrong target. That is a **`ValueError`**.

If none of those fits, the arguments are fine and the work could not be done. That is an **`ExecutionError`**, and its `kind` says why. A wrong shapes graph is a **`ShapeError`**, a SPARQL endpoint that failed is an **`EndpointError`**, and an optional package that is missing is an **`ImportError`**.

A test for questions 2 and 3: if the value is *parsed*, a failure is a `FormatError`; if it is *chosen or combined*, a failure is a `ValueError`. The same bad base direction is a `ValueError` when it is passed to `literal(..., direction="up")`, because the direction is chosen from `ltr` and `rtl`. It is a `FormatError` when it appears in text that is being parsed (`"x"@sv--up`), because the text is not in the format.

The `kind` then says which cause, and the next sections list them with an example of each.

## 3. `FormatError`: the text or data is malformed

`FormatError` has a `kind` for the language or format that is malformed:

| `kind` | The malformed thing |
| --- | --- |
| `sparql` | SPARQL query or update text, an `sqe:` tree, a `VERSION` line |
| `srl` | SRL text, an `srl:` tree, a rule set built with something that is not a rule body element, a bad `DATA` block, a rule set that breaks §4.2 |
| `rdf` | an RDF document that cannot be parsed, the text of a term (an IRI, a language tag, a lexical form that does not fit its datatype), a list that is not well formed, tied or missing ordinals in a tree |
| `manchester` | Manchester Syntax text, a `manch:` tree, a SWRL `Rule:` frame |
| `owl` | RDF that is not valid OWL when it is written as Manchester Syntax |

It carries `line` and `col` when the parser knows where the problem is, and `context` says which input it was (for example "language tag" or "Turtle document").

Below, one malformed input of each of the most common kinds is read, and the class, kind, context and line are printed.

In [ ]:
show("SPARQL text that is not SPARQL", lambda: sparql_to_tree("this is not sparql"))
show("a query that contradicts its VERSION line", lambda: g.query('VERSION "1.1" SELECT ?s WHERE { ?s ?p <<( ?s ?p ?o )>> }'))
show("a Turtle document with a syntax error", lambda: StarLayerGraph().parse(data="<a> <b>", format="turtle"))
show("text that is not exactly one term", lambda: parse_term("4 5"))
show("a lexical form that does not fit its datatype", lambda: to_python(parse_term('"abc"^^xsd:integer', {"xsd": str(XSD)})))
show("an unfinished SRL rule", lambda: srl.parse_ruleset("RULE { ?x"))
show("Manchester text with a syntax error", lambda: manchester_to_tree("Prefix: : <http://example.org/>\nClass: A SubClassOf:"))
show("a SWRL rule in Manchester text (not supported)", lambda: manchester_to_tree("Prefix: : <http://example.org/>\nRule: Person(?x)"))

SPARQL text that is not SPARQL
   FormatError, kind='sparql', context=None line 1
   expected SELECT, ASK, CONSTRUCT or DESCRIBE (found 'this')
a query that contradicts its VERSION line
   FormatError, kind='sparql', context='VERSION'
   SPARQL query declares VERSION '1.1' but uses a triple term, which '1.1' conformance excludes (SPARQL
a Turtle document with a syntax error
   FormatError, kind='rdf', context='Turtle document' line 1
   Parser error at line 1 between columns 1 and 4: No scheme found in an absolute IRI (line 1)
text that is not exactly one term
   FormatError, kind='rdf', context='term text'
   not exactly one RDF term in Turtle syntax
a lexical form that does not fit its datatype
   FormatError, kind='rdf', context='xsd:integer literal'
   'abc' is not a valid integer
an unfinished SRL rule
   FormatError, kind='srl', context=None line 1
   expected a predicate (found end of input)
Manchester text with a syntax error
   FormatError, kind='manchester', context=None
   u

## 4. `ExecutionError`: the work could not be done

The input is valid, but running it failed or reached a limit. The `kind` says why:

| `kind` | What failed |
| --- | --- |
| `query` | Oxigraph could not evaluate a query or update whose text parsed (for example a custom function that was not passed) |
| `rule` | a rule set could not be run: it cannot be ordered (stratification), or a rule failed |
| `import` | a document that an import needs could not be loaded, or was never loaded |
| `limit` | a size or complexity limit was reached: nesting that is too deep, SHACL `max_iterations` or `max_triples`, a graph too symmetric to canonicalise |
| `unsupported` | valid input that StarLayer, or the notation it writes, cannot handle: a Manchester construct with no counterpart, an entailment regime that is not registered, a query form `relevant_subgraph` cannot handle |
| `timeout`, `inconsistent`, `inference` | OWL 2 DL reasoning: the time budget ran out, the data is inconsistent, or the reasoner could not run (see [02.07](02.07-owl-dl-reasoning.ipynb)) |
| `expression`, `recursion`, `constraint-query` | SHACL: a node expression cannot be evaluated, a shape reaches itself through `sh:not`, a `sh:sparql` query reported a failure (see [05.02](05.02-starlayer-shacl-engine.ipynb)) |
| `internal` | a state that should be impossible: a bug in StarLayer |

An `internal` error says so in its message and gives the address for reporting it (`error.report_url` holds the address, so an application can show it).

Below, one example each of `query`, `limit`, `rule`, `import` and `unsupported` (twice).

In [ ]:
show("a custom function that was not passed", lambda: list(g.query("SELECT (<urn:fn:x>(?s) AS ?y) WHERE { ?s ?p ?o }")))
show("a query nested far too deeply (kind limit)", lambda: sparql_to_tree("SELECT ?x WHERE " + "{ " * 3000 + "?x ?p ?o" + " }" * 3000))
show("a rule set that cannot be ordered", lambda: srl.parse_ruleset("PREFIX ex: <http://example.org/> RULE { ?x ex:f ex:y } WHERE { ?x a ex:C . NOT { ?x ex:f ex:y } }").plan())
show("a rule set with imports that were never resolved", lambda: srl.parse_ruleset("IMPORTS <http://example.org/rules> RULE { ?x <urn:p> ?y } WHERE { ?x <urn:q> ?y }").infer(g))

labelled = StarLayerGraph()
labelled.parse(data='@prefix : <http://x/> . @prefix owl: <http://www.w3.org/2002/07/owl#> . @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> . :A a owl:Class ; rdfs:label "x"@ar--rtl .', format="turtle")
show("an entailment name that is not a registered regime", lambda: g.infer(entailment="nonsense"))
show("a graph that Manchester Syntax cannot write", lambda: labelled.serialize(format="manchester"))

a custom function that was not passed
   ExecutionError, kind='query', context='Oxigraph query'
   The custom function <urn:fn:x> is not supported
a query nested far too deeply (kind limit)
   ExecutionError, kind='limit', context='SPARQL parse nesting depth'
   SPARQL expression or pattern nested too deeply to parse
a rule set that cannot be ordered
   ExecutionError, kind='rule', context='stratification'
   SRL: stratification condition violated - a recursive dependency involves a closed edge, so this rule
a rule set with imports that were never resolved
   ExecutionError, kind='import', context='IMPORTS <http://example.org/rules>'
   SRL: IMPORTS <http://example.org/rules> is not resolved: the rule set was not resolved before it ran
an entailment name that is not a registered regime
   ExecutionError, kind='unsupported', context='entailment regime'
   Unsupported entailment regime(s) 'nonsense'; the available regimes are ['http://www.w3.org/ns/entail
a graph that Manchester Syntax c

## 5. `ShapeError`: the shapes are wrong

`ShapeError` is about the shapes graph, not the data. It is raised when the shapes cannot be read or compiled, and `MetaShaclError` (a `ShapeError`) is raised by the meta check, with the problems it found in `problems` and the full report in `report_text`.

Below, a shapes graph with an invalid `sh:datatype` is checked.

In [ ]:
bad_shapes = StarLayerGraph()
bad_shapes.parse(data="@prefix sh: <http://www.w3.org/ns/shacl#> . @prefix ex: <http://example.org/> . ex:S a sh:NodeShape ; sh:property [ sh:path ex:p ; sh:datatype 5 ] .", format="turtle")
try:
    meta_validate(bad_shapes)
except ShapeError as error:
    print(type(error).__name__)
    print("problems:", error.problems)

MetaShaclError
problems: ('the property shape of ex:S for ex:p, sh:datatype: violates sh:or',)


## 6. Wrong arguments stay `ValueError` and `TypeError`

A call that is given an argument it cannot use raises a `ValueError` (a value outside what the call defines, options that do not go together, an object in a state the call refuses) or a `TypeError` (the wrong Python type). These are not part of the family, because the cause is in the call and not in the data.

Below, a literal with both a language and a datatype, a base direction that is not `ltr` or `rtl`, the same bad direction inside text that is parsed (a `FormatError`, for contrast), a term that is not text, and a graph with no query to write.

In [ ]:
show("a literal with a language and a datatype", lambda: literal("5", XSD.integer, lang="sv"))
show("a base direction that is not ltr or rtl", lambda: literal("x", lang="sv", direction="up"))
show("the same direction inside text that is parsed", lambda: parse_term('"x"@sv--up'))
show("parse_term given a number", lambda: parse_term(5))
show("tree_to_sparql on a graph with no query", lambda: tree_to_sparql(StarLayerGraph()))

a literal with a language and a datatype
   ValueError, kind=None, context=None
   a literal has a language tag or a datatype, not both: lang='sv' with datatype <http://www.w3.org/200
a base direction that is not ltr or rtl
   ValueError, kind=None, context=None
   direction must be 'ltr' or 'rtl', not 'up'
the same direction inside text that is parsed
   FormatError, kind='rdf', context='term text'
   not exactly one RDF term in Turtle syntax
parse_term given a number
   TypeError, kind=None, context=None
   parse_term() needs a str, not int
tree_to_sparql on a graph with no query
   ValueError, kind=None, context=None
   the graph holds no root (no sqe:Query, sqe:UpdateRequest or srl:RuleSet): nothing to write


## 7. Catching errors

Catch the narrowest class that you can handle. `except StarLayerError` is the catch-all for the library. When a handler is for one `kind` of an `ExecutionError` or `FormatError`, call `only_kind(error, "kind")` first: it raises the error again if its kind is a different one, so a handler never swallows a cause it was not written for.

Below, three texts go through one function that handles a malformed text, a text that is nested too deeply (a `limit`), and any other StarLayer error.

In [ ]:
def read(text):
    try:
        sparql_to_tree(text)
    except FormatError as error:
        print("malformed", error.kind, "at line", error.line)
    except ExecutionError as error:
        only_kind(error, "limit")  # handle only this kind; any other kind is raised again
        print("too deep to read")
    except StarLayerError as error:
        print("another StarLayer error:", type(error).__name__)
    else:
        print("read")


read("SELECT ?s WHERE { ?s ?p ?o }")
read("not a query")
read("SELECT ?x WHERE " + "{ " * 3000 + "?x ?p ?o" + " }" * 3000)

read
malformed sparql at line 1
too deep to read


## 8. What changed from the earlier release

If your code caught one of the old errors, this is what to catch now.

| You used to catch | Now catch |
| --- | --- |
| `SyntaxError` from `parse()` | `FormatError`, kind `rdf` |
| `SyntaxError` or `SparqlSyntaxError` from a query or update text | `FormatError`, kind `sparql` |
| `RuntimeError` from `query()` or `update()` | `ExecutionError`, kind `query` |
| `SQEError`, `SQEValidationError`, `SPARQL12ConformanceError` | `FormatError`, kind `sparql`. A tree that breaks the `sqe:` shapes is no longer an error at parse time: `validate_sparql_tree()` reports it. |
| `ValueError` from `parse_term`, `to_python`, or an invalid IRI or language tag | `FormatError`, kind `rdf` (and `TypeError` for a value that is not text) |
| `ValueError` from `graph.items()` for a bad list | `FormatError`, kind `rdf` |
| `SRLParseError`, `SRLDecodeError`, `SRLEvalError` | `FormatError`, kind `srl`. A rule set is no longer checked when it is parsed: `validate_sparql_tree()` reports the problems and running the rule set raises. |
| `StratificationError` | `ExecutionError`, kind `rule` |
| `SRLImportError`, `SRLImportsNotSupportedError` | `ExecutionError`, kind `import` (or `FormatError`, kind `srl`, when the imported text is not a rule set) |
| `ManchesterSyntaxError` | `FormatError`, kind `manchester` |
| a graph Manchester Syntax could not write | `ExecutionError`, kind `unsupported` (or `FormatError`, kind `owl`, for RDF that is not valid OWL) |
| `InvalidTripleTermError`, `BasicEncodingError` | `FormatError`, kind `rdf` |
| `CanonicalizationComplexityError` | `ExecutionError`, kind `limit` |
| `EntailmentRewriteError`, `SqeTextTooDeepError` | `ExecutionError`, kind `unsupported` or `limit` |
| `ReasoningTimeoutError`, `InconsistentOntologyError` | `ExecutionError`, kind `timeout` or `inconsistent` (`inference` when the reasoner could not run) |
| a missing OWL reasoner package as `RuntimeError` | `ImportError` |
| `NotImplementedError` or `ValueError` for an entailment name that is not a registered regime | `ExecutionError`, kind `unsupported` |
| `except ValueError` for a wrong shapes graph | `ShapeError` (it is no longer a `ValueError`) |
| the tuple `(conforms, report_graph, report_text)` from a validator | a `ValidationReport` with `.conforms`, `.report_graph` and `.report_text` |

The functions were renamed in the same release: `sparql_to_tree`, `tree_to_sparql` and `validate_sparql_tree` for SPARQL, updates and SRL, and `manchester_to_tree`, `tree_to_manchester`, `tree_to_owl` and `validate_manchester_tree` for Manchester Syntax.

## 9. Decisions and open questions

### 9.1 Decided in the review

| # | Question | Decision |
| --- | --- | --- |
| 1 | Should the SHACL limits (`max_iterations`, `max_triples`) share a kind with the RDFC cap? | Yes: kind `limit` for every cap. `timeout` stays its own kind. |
| 2 | Is term text passed as an argument a `FormatError` or a `ValueError`? | `FormatError`: text that is turned into graph form needs a proper format. A choice from a fixed set (a base direction) stays `ValueError`. |
| 3 | An entailment name that is not a registered regime | `ExecutionError`, kind `unsupported`, for any text that is not registered (a typo, or D-Entailment). Names are an open vocabulary of IRIs, so a custom regime can be added later. |
| 4 | Where does `ValueError` end and `unsupported` begin? | Valid input that the library cannot handle is `unsupported`; a value outside what the call defines, contradictory options, or a wrong target stays `ValueError`. |
| 5 | A rule body element that is not one of the known elements | `FormatError`, kind `srl`: the rule set is malformed. |
| 6 | A call on a closed `Materialization` | `ValueError`: an object in a state the call refuses. |
| 7 | Impossible states | `ExecutionError`, kind `internal`, and the message says it is a bug in StarLayer and where to report it. |
| 8 | File and operating-system errors | Not wrapped. The one JSON-LD case, where the format cannot hold a graph, is `unsupported`. |
| 9 | Deep nesting: `FormatError` or `ExecutionError`? | `ExecutionError`, merged into kind `limit`. The text is valid and the library has a cap. |
| 10 | Kinds that could be merged | `term` merged into `rdf`. `manchester` and `owl` stay apart, and `import` stays. |
| 11 | Imports that cannot be loaded | One rule for every path (OWL graphs, SHACL shapes graphs, SRL): after resolution each import IRI must be the subject of a triple, or the call raises `ExecutionError`, kind `import`. A loader that returns nothing is an error. `ignore_imports=True` skips the check. Being built. |

### 9.2 Still open

12. **`EndpointError` has two bases** (`StarLayerError` and `OSError`), so callers that catch `OSError` for a network failure keep working. The alternative is an `ExecutionError` of kind `endpoint` with a `status`. *Recommendation:* keep.
13. **`context` is free text.** It is not a registry, so a reader can only match on the text. *Recommendation:* keep for now; add a registry only if callers start matching on it.
14. **`ShapeError` is no longer a `ValueError`.** This is the change most likely to break existing callers that wrote `except ValueError` around shape code. *Recommendation:* keep; it is listed in section 8.
15. **A shapes graph's `sh:entailment` that names an unregistered regime** still raises `ShapeCompileError`. Q3 says `unsupported`. *Recommendation:* change it to `unsupported`, so `infer()`, `query()` and a shapes graph give one answer.
16. **Other `ValueError` sites that look like "valid input the library cannot handle"** (Q4's test): `export_rdf11` ("cannot hold named graphs"), `qnames` ("cannot split IRI"), an OWL regime given to the RDF/RDFS query rewrite. *Recommendation:* change them to `unsupported`.
17. **A `TypeError` for "a SELECT result has no rows"** is a call on an object in the wrong state, so by the ladder it is a `ValueError`. *Recommendation:* change it.
18. **The loader parameter has three names** (`loader=`, `shapes_graph_loader=`, `imports_loader=`). *Recommendation:* decide later whether to unify them; renaming breaks callers.
19. **Which entry points resolve imports by default.** The rule is built for OWL graphs, SHACL shapes graphs and SRL. Whether `infer()` and `query()` for the OWL regimes should also raise on an unresolved `owl:imports` depends on what it would break in existing ontologies. *Recommendation:* decide after the build report lists the cases.
20. **Housekeeping.** The generated API docs (`docs/generated/api-*.generated.md`) and the code graph still name removed classes and need regenerating. Two tests of the one-off guide renumbering tool (`tests/repo/test_renumber_guides.py`) fail, and the tool can be retired.